# 3.3 Lab - Data Skipping and Liquid Clustering
**Lesson:** Data Skipping and Liquid Clustering

Delta stores **min/max statistics per file** for the first 32 columns. A filter such as `customer_id = 4242` can skip every file whose `[min, max]` range does not contain 4242, but **only if the data is laid out so that ranges don't overlap**. You will compare four layouts of the same table:

| Layout | How |
|---|---|
| Random (no layout) | as written |
| Partitioned | `PARTITIONED BY (store_id)` |
| Z-ORDER | `OPTIMIZE ... ZORDER BY (customer_id)` |
| **Liquid Clustering** | `CLUSTER BY (customer_id)` + `OPTIMIZE` |

Replace each `<FILL_IN>` and run the CHECK cells. The solution is in `03-performance-optimization/solutions/`.

In [ ]:
from adelab.classroom import setup
from adelab import perf
from pyspark.sql import functions as F

DA = setup("3.3", reset=True)
# Small target files so this laptop-sized table still has many files to skip (production default ~1 GB).
spark.conf.set("spark.databricks.delta.optimize.maxFileSize", 1 * 1024 * 1024)

source = (spark.range(3_000_000)
          .withColumn("customer_id", (F.rand(seed=1) * 100_000).cast("int"))
          .withColumn("store_id", (F.rand(seed=2) * 50).cast("int"))
          .withColumn("amount", F.round(F.rand(seed=3) * 500, 2))
          .withColumn("tx_date", F.expr("date'2026-01-01' + cast(rand(4) * 365 AS INT)")))

## Step 1 - Baseline: random layout
Write the data as 32 files in random order, then measure a selective query.

In [ ]:
source.repartition(32).write.saveAsTable("tx_random")
QUERY = "SELECT * FROM {t} WHERE customer_id BETWEEN 4200 AND 4300"
baseline = perf.run(spark.sql(QUERY.format(t="tx_random")), "random")
display(perf.compare(baseline))
print(perf.table_files(spark, "tx_random"))

Every file's `[min, max]` of `customer_id` covers almost the whole domain, so **every file must be read**. Look at the statistics Delta stored for the first files:

![Random layout: every file's min/max range covers the query value, so all files are read. Clustered layout: ranges are disjoint, so only one file is read](../images/3.3-data-skipping-min-max.svg)

**How to read it:** each bar is one file's `[min, max]` of `customer_id`; the dotted line is the query's filter.
Delta can skip a file only when the line misses its bar. The `min/max` values printed below match the top panel;
Steps 2 and 3 turn the table into the bottom panel, and `files_read` falls accordingly.

In [ ]:
import json
stats = (spark.read.json(f"{spark.sql('DESCRIBE DETAIL tx_random').first()['location']}/_delta_log/*.json")
         .filter("add IS NOT NULL").select("add.stats").limit(3).collect())
for s in stats:
    st = json.loads(s.stats)
    print("customer_id min/max:", st["minValues"]["customer_id"], st["maxValues"]["customer_id"])

## Step 2 - Z-ORDER
Create `tx_zorder` as a copy of `tx_random`, then Z-ORDER it by `customer_id`.

In [ ]:
spark.sql("CREATE TABLE tx_zorder AS SELECT * FROM tx_random")
spark.sql("<FILL_IN>")
zorder = perf.run(spark.sql(QUERY.format(t="tx_zorder")), "zorder")
display(perf.compare(baseline, zorder))

In [ ]:
# CHECK step 2
hist = spark.sql("DESCRIBE HISTORY tx_zorder").filter("operation = 'OPTIMIZE'").first()
assert hist is not None and "customer_id" in hist["operationParameters"]["zOrderBy"],     "run OPTIMIZE tx_zorder ZORDER BY (customer_id)"
assert zorder["files_read"] < baseline["files_read"] / 4, "Z-ORDER should let the query skip most files"
print("Step 2 passed")

## Step 3 - Liquid Clustering
Create `tx_liquid` **with clustering keys** `customer_id`, load it from `tx_random`, and run `OPTIMIZE` (which clusters incrementally).

Differences from Z-ORDER:
- Keys are table metadata (`CLUSTER BY`), not an argument you must repeat on every OPTIMIZE
- **Incremental**: OPTIMIZE only reclusters what is needed, while Z-ORDER rewrites everything
- Keys can be **changed** without rewriting data (`ALTER TABLE ... CLUSTER BY`)
- Not combinable with partitioning or Z-ORDER. Replaces both for new tables

In [ ]:
spark.sql("CREATE TABLE tx_liquid <FILL_IN> AS SELECT * FROM tx_random")
spark.sql("<FILL_IN>")
liquid = perf.run(spark.sql(QUERY.format(t="tx_liquid")), "liquid")
display(perf.compare(baseline, zorder, liquid))

In [ ]:
# CHECK step 3
d = spark.sql("DESCRIBE DETAIL tx_liquid").first()
assert list(d["clusteringColumns"]) == ["customer_id"], "tx_liquid must be clustered by customer_id"
assert liquid["files_read"] < baseline["files_read"] / 4, "did you run OPTIMIZE?"
print("Step 3 passed")

## Step 4 - Partitioning is the wrong tool here
Partition a copy by `store_id` and run **the same customer query**. Partition pruning only helps filters **on the partition column**, and there are now 50 directories of smaller files.

![Partitioning, Z-ORDER and Liquid Clustering compared: how each lays out files and when to use it](../images/3.3-layout-comparison.svg)

**How to read it:** all three aim to make per-file ranges narrow for the columns you filter on. Partitioning does it with directories
and only for its own column; Z-ORDER and Liquid Clustering do it inside files, so they help `customer_id` filters.
The two partitioned rows in the next table show both sides: pruning on `store_id`, none on `customer_id`.

In [ ]:
spark.sql("CREATE TABLE tx_partitioned PARTITIONED BY (store_id) AS SELECT * FROM tx_random")
partitioned = perf.run(spark.sql(QUERY.format(t="tx_partitioned")), "partitioned by store_id")
store_query = perf.run(spark.sql("SELECT * FROM tx_partitioned WHERE store_id = 7"), "partitioned, filter on store_id")
display(perf.compare(baseline, zorder, liquid, partitioned, store_query))
perf.plot([baseline, zorder, liquid, partitioned, store_query], "files_read", "Files read per layout",
          highlight="liquid", note="Liquid Clustering on the filter column");

Look for the short bars: `zorder` and `liquid` skip most files for the customer filter, while `partitioned by store_id` reads about as many as `random` unless the filter is on `store_id`.

## Step 5 - Workloads change: switch the clustering key
Analysts now filter mostly by `tx_date`. Change `tx_liquid`'s clustering key to `tx_date` **without rewriting the table** in a separate step, then `OPTIMIZE` and measure a date query.

In [ ]:
DATE_QUERY = "SELECT * FROM {t} WHERE tx_date = date'2026-06-15'"
before = perf.run(spark.sql(DATE_QUERY.format(t="tx_liquid")), "date query, clustered by customer_id")
spark.sql("<FILL_IN: change the clustering key>")
spark.sql("<FILL_IN: recluster all existing data>")
after = perf.run(spark.sql(DATE_QUERY.format(t="tx_liquid")), "date query, clustered by tx_date")
display(perf.compare(before, after))

In [ ]:
# CHECK step 5
assert list(spark.sql("DESCRIBE DETAIL tx_liquid").first()["clusteringColumns"]) == ["tx_date"]
assert after["files_read"] < before["files_read"], "the date query should now skip files"
print("Step 5 passed")

## Step 6 - Which columns have statistics?
Only the first `delta.dataSkippingNumIndexedCols` columns (default 32) get statistics. Wide tables should list their filter columns explicitly with `delta.dataSkippingStatsColumns`. Clustering keys must have statistics. Set stats collection on `tx_liquid` to just `customer_id, tx_date, store_id`:

In [ ]:
spark.sql("ALTER TABLE tx_liquid SET TBLPROPERTIES (<FILL_IN>)")
display(spark.sql("SHOW TBLPROPERTIES tx_liquid").filter("key LIKE 'delta.dataSkipping%'"))

In [ ]:
# CHECK step 6
props = {r.key: r.value for r in spark.sql("SHOW TBLPROPERTIES tx_liquid").collect()}
cols = {c.strip() for c in props.get("delta.dataSkippingStatsColumns", "").split(",") if c.strip()}
assert cols == {"customer_id", "tx_date", "store_id"}, f"delta.dataSkippingStatsColumns = {cols or 'not set'}"
print("Step 6 passed. New files get statistics for these 3 columns only. "
      "Existing files keep their old statistics until rewritten (ANALYZE TABLE ... COMPUTE DELTA STATISTICS recomputes them on Databricks).")

## Summary
| Choose | When |
|---|---|
| **Liquid Clustering** (or `CLUSTER BY AUTO` on Databricks) | Default for new tables, high-cardinality filter columns, changing access patterns |
| Partitioning | Very large tables (TB+), a low-cardinality column used in almost every filter, or data lifecycle by partition |
| Z-ORDER | Existing partitioned tables on older runtimes |

In [ ]:
DA.cleanup()